In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [7]:
spark = SparkSession.builder.appName("UdfApp") \
        .master("local[4]") \
        .config("spark.dynamicAllocation.enabled", "false") \
        .config("spark.sql.adaptive.enabled", "false") \
        .getOrCreate()

In [8]:
sc = spark.sparkContext

In [5]:
spark

<SparkContext master=local[4] appName=UdfApp>

25/04/06 16:53:36 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [9]:
cabsDF = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv("/home/luffy/Documents/Spark/Data/Cabs.csv")

In [10]:
cabsDF.createGlobalTempView("Cabs")
cabsDF.show(truncate=False)

+---------+--------------------+------------------------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+---------------+--------------------------+--------------------------------------------+---------------+
|CabNumber|VehicleLicenseNumber|Name                    |LicenseType     |Active|PermitLicenseNumber|VehicleVinNumber |WheelchairAccessible|VehicleYear|VehicleType|TelephoneNumber|Website                   |Address                                     |LastDateUpdated|
+---------+--------------------+------------------------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+---------------+--------------------------+--------------------------------------------+---------------+
|T802127C |C19641              |ABCON INC.              |OWNER MUST DRIVE|YES   |NULL               |5TDBK3EH0DS268018|NULL                |2016       |NULL       |(718)438-1100  |NULL         

In [14]:
def convertCase(string):
    result = ""
    
    nameWordsArray = string.split(",")
    for nameWord in nameWordsArray:
        result = result + nameWord[0:1].upper() + nameWord[1:len(nameWord)].lower() + ","
    
    result = result[0:len(result)-1]
    return result


In [15]:
convertCaseUdf = udf(lambda string: convertCase(string), StringType())

In [16]:
cabsDF.select("Name", convertCaseUdf(col("Name")).alias("Name_ConvertedCase")).show()

+--------------------+--------------------+
|                Name|  Name_ConvertedCase|
+--------------------+--------------------+
|          ABCON INC.|          Abcon inc.|
| ACCEPTABLE TAXI LLC| Acceptable taxi llc|
|      ALLIS CAB CORP|      Allis cab corp|
|       BENE CAB CORP|       Bene cab corp|
|   BOULOS TAXI CORP.|   Boulos taxi corp.|
|     CACERES,JAIME,P|     Caceres,Jaime,P|
|CALCIUM ONE SERVI...|Calcium one servi...|
|     CHARLES,WILBERT|     Charles,Wilbert|
|      CHAWKI,MICHAEL|      Chawki,Michael|
|CHRYSOVALANTOU CORP,|Chrysovalantou corp,|
|     COFI BOAT CORP.|     Cofi boat corp.|
| DEKEL TAXI CAB CORP| Dekel taxi cab corp|
|FLORIAN & ROBERT INC|Florian & robert inc|
|       GART CAB CORP|       Gart cab corp|
|    GAUTHIER,JACQUES|    Gauthier,Jacques|
|GEORGAKOPOULOS, G...|Georgakopoulos, g...|
|      GUJAR CAB CORP|      Gujar cab corp|
|     HUEZO, SALVADOR|     Huezo, salvador|
|   JEAN-PIERRE,SERGE|   Jean-pierre,Serge|
|      JETS CAB CORP.|      Jets

In [17]:
spark.udf.register("convertCaseSqlUdf", convertCase, StringType())

<function __main__.convertCase(string)>

In [19]:
spark.sql("""
SELECT Name, convertCaseSqlUdf(Name) AS Name_ConvertedCase FROM global_temp.Cabs
""").show()

+--------------------+--------------------+
|                Name|  Name_ConvertedCase|
+--------------------+--------------------+
|          ABCON INC.|          Abcon inc.|
| ACCEPTABLE TAXI LLC| Acceptable taxi llc|
|      ALLIS CAB CORP|      Allis cab corp|
|       BENE CAB CORP|       Bene cab corp|
|   BOULOS TAXI CORP.|   Boulos taxi corp.|
|     CACERES,JAIME,P|     Caceres,Jaime,P|
|CALCIUM ONE SERVI...|Calcium one servi...|
|     CHARLES,WILBERT|     Charles,Wilbert|
|      CHAWKI,MICHAEL|      Chawki,Michael|
|CHRYSOVALANTOU CORP,|Chrysovalantou corp,|
|     COFI BOAT CORP.|     Cofi boat corp.|
| DEKEL TAXI CAB CORP| Dekel taxi cab corp|
|FLORIAN & ROBERT INC|Florian & robert inc|
|       GART CAB CORP|       Gart cab corp|
|    GAUTHIER,JACQUES|    Gauthier,Jacques|
|GEORGAKOPOULOS, G...|Georgakopoulos, g...|
|      GUJAR CAB CORP|      Gujar cab corp|
|     HUEZO, SALVADOR|     Huezo, salvador|
|   JEAN-PIERRE,SERGE|   Jean-pierre,Serge|
|      JETS CAB CORP.|      Jets